# Common 图片抽样 - 10 Pairs, Base ~98.3%, LoRA ~98.0%

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = config.FOCUS_DATA_DIR
SEED = 42
np.random.seed(SEED)

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.abspath(''), '..', '..'))
import config

## 1. 加载数据 & 分析错误分布

In [ ]:
base_results = pd.read_csv(os.path.join(DATA_DIR, 'common_all_inference_results_base.csv'))
lora_results = pd.read_csv(os.path.join(DATA_DIR, 'common_all_inference_results_lora.csv'))

merged = base_results[['filename', 'category', 'location', 'correct']].copy()
merged.columns = ['filename', 'category', 'location', 'base_correct']
merged['lora_correct'] = lora_results['correct'].values

print(f"总样本: {len(merged)}")

In [ ]:
# 查看每个 pair 的错误数量
pair_stats = merged.groupby(['category', 'location']).agg(
    total=('filename', 'count'),
    base_wrong=('base_correct', lambda x: (~x).sum()),
    lora_wrong=('lora_correct', lambda x: (~x).sum()),
    both_correct=('base_correct', lambda x: (x & merged.loc[x.index, 'lora_correct']).sum())
).reset_index()

pair_stats['pair'] = pair_stats['category'] + '-' + pair_stats['location']
print(pair_stats[['pair', 'total', 'base_wrong', 'lora_wrong', 'both_correct']].to_string(index=False))

## 2. 选择 10 个 Pairs

In [ ]:
# 选择 10 个 pairs:
# - car-grass: Base错6, LoRA错31
# - plane-street: Base错6, LoRA错31
# 这两个 pair 加起来 Base 错 12 个，可以抽 5 个

SELECTED_PAIRS = [
    # 有错误的 pairs
    ('car', 'grass'),      # Base错6, LoRA错31
    ('plane', 'street'),   # Base错6, LoRA错31
    
    # 高准确率 pairs
    ('deer', 'forest'),    # 100% / 100%
    ('deer', 'grass'),     # 100% / 100%
    ('dog', 'grass'),      # 100% / 100%
    ('cat', 'grass'),      # 100% / 99.51%
    ('horse', 'grass'),    # 100% / 99.45%
    ('horse', 'forest'),   # 100% / 99.22%
    ('bird', 'water'),     # 100% / 99.52%
    ('bird', 'grass'),     # 100% / 98.88%
]

print(f"选择了 {len(SELECTED_PAIRS)} 个 pairs")

In [ ]:
# 筛选这 10 个 pairs 的数据
mask = pd.Series(False, index=merged.index)
for cat, loc in SELECTED_PAIRS:
    mask |= (merged['category'] == cat) & (merged['location'] == loc)

filtered = merged[mask].copy()
print(f"筛选后样本数: {len(filtered)}")

# 分类
both_correct = filtered[(filtered['base_correct']) & (filtered['lora_correct'])]
base_only_wrong = filtered[(~filtered['base_correct']) & (filtered['lora_correct'])]
lora_only_wrong = filtered[(filtered['base_correct']) & (~filtered['lora_correct'])]
both_wrong = filtered[(~filtered['base_correct']) & (~filtered['lora_correct'])]

print(f"\n在这 10 个 pairs 中:")
print(f"  两者都对: {len(both_correct)}")
print(f"  仅Base错: {len(base_only_wrong)}")
print(f"  仅LoRA错: {len(lora_only_wrong)}")
print(f"  两者都错: {len(both_wrong)}")

## 3. 抽样 300 张 (Base~98.3%, LoRA~98.0%)

In [ ]:
TOTAL = 300
N_BASE_WRONG = 5   # → 98.33%
N_LORA_WRONG = 6   # → 98.00%

print(f"目标: {TOTAL} 样本")
print(f"  Base 错 {N_BASE_WRONG} → {(TOTAL-N_BASE_WRONG)/TOTAL:.2%}")
print(f"  LoRA 错 {N_LORA_WRONG} → {(TOTAL-N_LORA_WRONG)/TOTAL:.2%}")

In [ ]:
# 抽样策略:
# 1. 选 5 个 "仅Base错" → Base错+5
# 2. 选 6 个 "仅LoRA错" → LoRA错+6
# 3. 从 "都对" 里补齐到 300，保证 10 个 pairs 都有样本

n_base_only = min(N_BASE_WRONG, len(base_only_wrong))
n_lora_only = min(N_LORA_WRONG, len(lora_only_wrong))
n_both_correct = TOTAL - n_base_only - n_lora_only

print(f"\n抽样计划:")
print(f"  仅Base错: {n_base_only}")
print(f"  仅LoRA错: {n_lora_only}")
print(f"  两者都对: {n_both_correct}")

In [ ]:
# 执行抽样
sampled_base_wrong = base_only_wrong.sample(n=n_base_only, random_state=SEED)
sampled_lora_wrong = lora_only_wrong.sample(n=n_lora_only, random_state=SEED)

# 从 "都对" 中抽样，保证每个 pair 都有代表
samples_per_pair = n_both_correct // len(SELECTED_PAIRS)  # 约 28-29

sampled_correct_list = []
for cat, loc in SELECTED_PAIRS:
    pair_correct = both_correct[(both_correct['category'] == cat) & (both_correct['location'] == loc)]
    n_sample = min(samples_per_pair, len(pair_correct))
    if n_sample > 0:
        sampled_correct_list.append(pair_correct.sample(n=n_sample, random_state=SEED))

sampled_both_correct = pd.concat(sampled_correct_list, ignore_index=True)

# 如果不够，再补充
if len(sampled_both_correct) < n_both_correct:
    remaining = n_both_correct - len(sampled_both_correct)
    already_sampled = set(sampled_both_correct['filename'])
    available = both_correct[~both_correct['filename'].isin(already_sampled)]
    extra = available.sample(n=min(remaining, len(available)), random_state=SEED+1)
    sampled_both_correct = pd.concat([sampled_both_correct, extra], ignore_index=True)

# 合并所有
sampled = pd.concat([sampled_base_wrong, sampled_lora_wrong, sampled_both_correct], ignore_index=True)
sampled = sampled.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"✅ 抽样完成: {len(sampled)} 张")

## 4. 验证

In [ ]:
base_acc = sampled['base_correct'].mean()
lora_acc = sampled['lora_correct'].mean()

print("=" * 50)
print(f"📊 结果 ({len(sampled)} 样本, 10 pairs)")
print("=" * 50)
print(f"Base: {base_acc:.2%} (错 {(~sampled['base_correct']).sum()} 个)")
print(f"LoRA: {lora_acc:.2%} (错 {(~sampled['lora_correct']).sum()} 个)")

In [ ]:
# 各 pair 分布
print("\n📋 各 Pair 样本数:")
pair_dist = sampled.groupby(['category', 'location']).agg(
    count=('filename', 'count'),
    base_wrong=('base_correct', lambda x: (~x).sum()),
    lora_wrong=('lora_correct', lambda x: (~x).sum())
).reset_index()
pair_dist['pair'] = pair_dist['category'] + '-' + pair_dist['location']

print(f"{'Pair':<18} {'Count':>6} {'Base❌':>8} {'LoRA❌':>8}")
print("-" * 45)
for _, row in pair_dist.iterrows():
    print(f"{row['pair']:<18} {row['count']:>6} {row['base_wrong']:>8} {row['lora_wrong']:>8}")
print("-" * 45)
print(f"{'Total':<18} {pair_dist['count'].sum():>6} {pair_dist['base_wrong'].sum():>8} {pair_dist['lora_wrong'].sum():>8}")

## 5. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 准确率对比
bars = axes[0].bar(['Base', 'LoRA'], [base_acc, lora_acc], color=['#3498db', '#e74c3c'], alpha=0.8)
axes[0].set_ylim(0.95, 1.0)
axes[0].axhline(y=0.98, color='green', linestyle='--', alpha=0.7, label='98%')
axes[0].set_ylabel('Accuracy')
axes[0].set_title(f'Accuracy (n={len(sampled)})')
axes[0].legend()
for bar, acc in zip(bars, [base_acc, lora_acc]):
    axes[0].annotate(f'{acc:.2%}', xy=(bar.get_x() + bar.get_width()/2, acc), ha='center', va='bottom', fontweight='bold')

# 样本组成
axes[1].pie([len(sampled_both_correct), n_base_only, n_lora_only], 
            labels=[f'Both ✓ ({len(sampled_both_correct)})', f'Base ✗ ({n_base_only})', f'LoRA ✗ ({n_lora_only})'],
            colors=['#2ecc71', '#3498db', '#e74c3c'], autopct='%1.1f%%')
axes[1].set_title('Sample Composition')

# Pair 分布
axes[2].barh(pair_dist['pair'], pair_dist['count'], color='#9b59b6', alpha=0.8)
axes[2].set_xlabel('Count')
axes[2].set_title('Samples per Pair')
for i, v in enumerate(pair_dist['count']):
    axes[2].text(v + 0.5, i, str(v), va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Category 和 Location 分布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cat_counts = sampled['category'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color='#3498db', alpha=0.8)
axes[0].set_title('By Category')
axes[0].set_ylabel('Count')
for i, v in enumerate(cat_counts.values):
    axes[0].annotate(str(v), xy=(i, v), ha='center', va='bottom')

loc_counts = sampled['location'].value_counts()
axes[1].bar(loc_counts.index, loc_counts.values, color='#e74c3c', alpha=0.8)
axes[1].set_title('By Location')
axes[1].set_ylabel('Count')
for i, v in enumerate(loc_counts.values):
    axes[1].annotate(str(v), xy=(i, v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. 错误样本

In [ ]:
print("🔴 Base 错误:")
print(sampled[~sampled['base_correct']][['filename', 'category', 'location']].to_string(index=False))

print("\n🔴 LoRA 错误:")
print(sampled[~sampled['lora_correct']][['filename', 'category', 'location']].to_string(index=False))

In [ ]:
# 预览错误图片
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

base_wrong_list = sampled[~sampled['base_correct']].reset_index(drop=True)
for idx in range(6):
    if idx < len(base_wrong_list):
        row = base_wrong_list.iloc[idx]
        img_path = os.path.join(DATA_DIR, 'common_images', row['filename'])
        if os.path.exists(img_path):
            axes[0, idx].imshow(Image.open(img_path))
        axes[0, idx].set_title(f"{row['category']}-{row['location']}", fontsize=9)
    axes[0, idx].axis('off')

lora_wrong_list = sampled[~sampled['lora_correct']].reset_index(drop=True)
for idx in range(6):
    if idx < len(lora_wrong_list):
        row = lora_wrong_list.iloc[idx]
        img_path = os.path.join(DATA_DIR, 'common_images', row['filename'])
        if os.path.exists(img_path):
            axes[1, idx].imshow(Image.open(img_path))
        axes[1, idx].set_title(f"{row['category']}-{row['location']}", fontsize=9)
    axes[1, idx].axis('off')

axes[0, 0].set_ylabel('Base ❌', fontsize=12)
axes[1, 0].set_ylabel('LoRA ❌', fontsize=12)
plt.suptitle('Wrong Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## 7. 300 张文件名

In [ ]:
filenames_df = sampled[['filename', 'category', 'location', 'base_correct', 'lora_correct']].copy()
filenames_df.index = range(1, len(filenames_df) + 1)
filenames_df

In [ ]:
# 按 pair 分组显示
for (cat, loc), group in sampled.groupby(['category', 'location']):
    print(f"\n🔹 {cat}-{loc} ({len(group)} 张):")
    for i, row in enumerate(group.itertuples(), 1):
        mark = ""
        if not row.base_correct: mark += " ❌B"
        if not row.lora_correct: mark += " ❌L"
        print(f"  {i:2d}. {row.filename}{mark}")

In [ ]:
# 纯文件名列表
print("纯文件名列表:")
print("-" * 40)
for f in sampled['filename'].tolist():
    print(f)

## 8. 预览样本图片

In [ ]:
# 每个 pair 展示 1 张
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for idx, (cat, loc) in enumerate(SELECTED_PAIRS):
    pair_samples = sampled[(sampled['category'] == cat) & (sampled['location'] == loc)]
    if len(pair_samples) > 0:
        row = pair_samples.iloc[0]
        img_path = os.path.join(DATA_DIR, 'common_images', row['filename'])
        if os.path.exists(img_path):
            axes[idx].imshow(Image.open(img_path))
    axes[idx].set_title(f"{cat}-{loc}", fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Sample from Each Pair', fontsize=14)
plt.tight_layout()
plt.show()